# 02 — Reproducible verification of the paper's LLM performance numbers

**Mission**: `6_confirm_LLM_performance_number/missions.md`
**Database**: `/workspace/ICOCO.duckdb`
**Pre-requisite**: run `01_bronze_ingestion.ipynb` (creates `bronze.llm_ingredient`,
`bronze.llm_scientific`, `bronze.dirty_ingredient_country_mapping`).

This notebook re-derives, **from the bronze layer in pure SQL**, the four numbers the paper
reports for the two LLM tasks, and compares each one against the paper claim:

| Metric | Ingredient Standardization (Task A) | Taxonomic Extraction (Task B) |
|---|---|---|
| Self-Consistency (SC) | **98.41%** | **82.34%** |
| Input Coverage (IC)   | **97.17%** | **61.36%** |
| Adjusted Coverage (AC)| **99.30%** | **98.74%** |
| Grounding (Queryability) | **71.20%** (11,765 / 16,524) | **93.29%** (2,253 / 2,415) |

> Where a printed ratio disagrees with a printed percentage, we trust the **percentage**
> and verify the exact fraction that yields it (see the discrepancies section at the bottom).

## Definitions used (as implemented by the legacy pipeline the paper reports)

For an input string $t$, the LLM is queried 3 times with temperature = 0.5 →
three candidate lists $L_1, L_2, L_3$ (a cell may fail to parse → it is kept as a raw string):

* **Self-Consistency (SC)** — *at least 2 of the 3 lists are identical* after sorting
  ($\max\emph freq of the canonical form $\ge 2/3$). *Paper text: "SC, ≥2/3 identical outputs
  [31]".* Empty lists count as identical to each other.
* **Input Coverage (IC)** — the union of the runs contains **≥ 1 candidate that appears in
  ≥ 2 of the 3 runs** (frequency ≥ 2/3).
* **Adjusted Coverage (AC)** — **IC** **or** ≥ 2/3 of the runs return an *empty* list
  (the legacy `self_consistency.py` recovers coverage when the LLM "agrees to not know").
* **Grounding / Queryability** — for inputs that pass the aggregation (gold majority-vote
  `final_output` non-empty): *queryable* iff ≥ 1 candidate from the gold list is found in the
  reference database.

The exact SQL semantics (including the handling of truncated / non-JSON cells) mirror the
legacy `old_code/utils/self_consistency.py` + `gold_data_create` behaviour, which is what the
paper's numbers come from.

## 0. Setup (connect → close → reconnect)

In [1]:
%load_ext sql
%config SqlMagic.displaylimit = 200
%sql duckdb:////workspace/ICOCO.duckdb

Connecting to 'duckdb:////workspace/ICOCO.duckdb'

In [2]:
%sql --close duckdb:////workspace/ICOCO.duckdb

In [3]:
%sql duckdb:////workspace/ICOCO.duckdb

Connecting and switching to connection 'duckdb:////workspace/ICOCO.duckdb'

## 1. Bronze sanity check

In [4]:
%%sql
SELECT 'bronze.llm_ingredient' AS tbl, count(*) AS rows FROM bronze.llm_ingredient
UNION ALL SELECT 'bronze.llm_scientific', count(*) FROM bronze.llm_scientific
UNION ALL SELECT 'bronze.dirty_ingredient_country_mapping', count(*) FROM bronze.dirty_ingredient_country_mapping;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

tbl,rows
bronze.llm_ingredient,17017
bronze.llm_scientific,4280
bronze.dirty_ingredient_country_mapping,4280


## 2. Parsing layer (`silver.v_*_runs`)

Each raw cell is *whitespace-trimmed* like Python's `str.strip()` (DuckDB `trim()` alone does
**not** strip `\n`, so we use `regexp_replace(..., '^\s+|\s+$', '')`). A cell that is a valid
JSON array becomes a `value_list`; anything else is kept as a raw string (`norm_str`) — the
exact behaviour of `ast.literal_eval` in the legacy parser.

In [5]:
%%sql
CREATE OR REPLACE VIEW silver.v_ingredient_runs AS
WITH runs AS (
    SELECT custom_id::VARCHAR AS custom_id, 'o1' AS run_idx, output1::VARCHAR AS raw FROM bronze.llm_ingredient
    UNION ALL
    SELECT custom_id::VARCHAR, 'o2', output2::VARCHAR FROM bronze.llm_ingredient
    UNION ALL
    SELECT custom_id::VARCHAR, 'o3', output3::VARCHAR FROM bronze.llm_ingredient
),
parsed AS (
    SELECT
        custom_id, run_idx, raw,
            regexp_replace(raw, '^\s+|\s+$', '') AS ws,
        CASE WHEN json_valid(    regexp_replace(raw, '^\s+|\s+$', '')) AND starts_with(    regexp_replace(raw, '^\s+|\s+$', ''), '[') AND ends_with(    regexp_replace(raw, '^\s+|\s+$', ''), ']')
             THEN 'list' ELSE 'string' END AS parsed_kind,
        CASE WHEN json_valid(    regexp_replace(raw, '^\s+|\s+$', '')) AND starts_with(    regexp_replace(raw, '^\s+|\s+$', ''), '[') AND ends_with(    regexp_replace(raw, '^\s+|\s+$', ''), ']')
             THEN list_transform(from_json(    regexp_replace(raw, '^\s+|\s+$', ''), '["VARCHAR"]'), x -> lower(trim(x)))
             ELSE NULL END AS value_list,
        CASE WHEN json_valid(    regexp_replace(raw, '^\s+|\s+$', '')) AND starts_with(    regexp_replace(raw, '^\s+|\s+$', ''), '[') AND ends_with(    regexp_replace(raw, '^\s+|\s+$', ''), ']')
             THEN NULL ELSE lower(    regexp_replace(raw, '^\s+|\s+$', '')) END AS norm_str
    FROM runs
)
SELECT * FROM parsed;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [6]:
%%sql
CREATE OR REPLACE VIEW silver.v_scientific_runs AS
WITH runs AS (
    SELECT custom_id::VARCHAR AS custom_id, 'o1' AS run_idx, output1::VARCHAR AS raw FROM bronze.llm_scientific
    UNION ALL
    SELECT custom_id::VARCHAR, 'o2', output2::VARCHAR FROM bronze.llm_scientific
    UNION ALL
    SELECT custom_id::VARCHAR, 'o3', output3::VARCHAR FROM bronze.llm_scientific
),
parsed AS (
    SELECT
        custom_id, run_idx, raw,
            regexp_replace(raw, '^\s+|\s+$', '') AS ws,
        CASE WHEN json_valid(    regexp_replace(raw, '^\s+|\s+$', '')) AND starts_with(    regexp_replace(raw, '^\s+|\s+$', ''), '[') AND ends_with(    regexp_replace(raw, '^\s+|\s+$', ''), ']')
             THEN 'list' ELSE 'string' END AS parsed_kind,
        CASE WHEN json_valid(    regexp_replace(raw, '^\s+|\s+$', '')) AND starts_with(    regexp_replace(raw, '^\s+|\s+$', ''), '[') AND ends_with(    regexp_replace(raw, '^\s+|\s+$', ''), ']')
             THEN list_transform(from_json(    regexp_replace(raw, '^\s+|\s+$', ''), '["VARCHAR"]'), x -> lower(trim(x)))
             ELSE NULL END AS value_list,
        CASE WHEN json_valid(    regexp_replace(raw, '^\s+|\s+$', '')) AND starts_with(    regexp_replace(raw, '^\s+|\s+$', ''), '[') AND ends_with(    regexp_replace(raw, '^\s+|\s+$', ''), ']')
             THEN NULL ELSE lower(    regexp_replace(raw, '^\s+|\s+$', '')) END AS norm_str
    FROM runs
)
SELECT * FROM parsed;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


Cell-parse statistics (how many replicas are real JSON lists vs. raw strings).

In [7]:
%%sql
SELECT 'Task A' AS task, parsed_kind, count(*) AS n FROM silver.v_ingredient_runs GROUP BY parsed_kind
UNION ALL SELECT 'Task B', parsed_kind, count(*) FROM silver.v_scientific_runs GROUP BY parsed_kind;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

task,parsed_kind,n
Task A,string,21
Task A,list,51030
Task B,list,12179
Task B,string,661


## 3. Self-Consistency / Input Coverage / Adjusted Coverage

`SC` = the **same canonical list** (sorted, elements lower-cased, non-JSON cells sorted by
their characters — exactly like `sorted()`+`json.dumps` in the legacy code) appears in **≥ 2
of the 3 runs**. `IC` = ≥ 1 candidate with count ≥ 2 in the 3-run union (empty string
excluded). `AC` = IC **or** ≥ 2/3 of the runs are empty.

In [8]:
%%sql
-- TASK A · Ingredient Standardization: SC / IC / AC over N = (SELECT count(*) FROM silver.v_ingredient_runs)/3 inputs
WITH canon AS (
    SELECT custom_id, run_idx,
        CASE WHEN parsed_kind = 'list'
             THEN to_json(list_sort(value_list))
             WHEN len(coalesce(norm_str, '')) = 0 THEN '[]'
             ELSE to_json(list_sort(string_to_array(coalesce(norm_str, ''), '')))
        END AS canon
    FROM silver.v_ingredient_runs
),
per_row AS (
    SELECT custom_id, max(c) AS max_run_freq
    FROM (SELECT custom_id, canon, count(*) AS c FROM canon GROUP BY custom_id, canon)
    GROUP BY custom_id
),
sc_flags AS (
    SELECT custom_id, CASE WHEN max_run_freq >= 2 THEN 1 ELSE 0 END AS sc
    FROM per_row
),
elems AS (
    SELECT custom_id, elem
    FROM (
        SELECT custom_id, parsed_kind, value_list, norm_str
        FROM silver.v_ingredient_runs
    ) p
    CROSS JOIN UNNEST(
        CASE WHEN p.parsed_kind = 'list' THEN p.value_list
             WHEN len(coalesce(p.norm_str, '')) = 0 THEN []
             ELSE string_to_array(coalesce(p.norm_str, ''), '') END
    ) AS t(elem)
),
elem_counts AS (
    SELECT custom_id, elem, count(*) AS cnt FROM elems GROUP BY custom_id, elem
),
ic_flags AS (
    SELECT custom_id, max(CASE WHEN cnt >= 2 AND elem <> '' THEN 1 ELSE 0 END) AS ic
    FROM elem_counts GROUP BY custom_id
),
empty_flags AS (
    SELECT custom_id,
           sum(CASE WHEN (parsed_kind = 'list' AND len(value_list) = 0)
                     OR (parsed_kind = 'string' AND len(coalesce(norm_str, '')) = 0)
                    THEN 1 ELSE 0 END) AS empty_cnt
    FROM silver.v_ingredient_runs GROUP BY custom_id
)
SELECT
    count(*)                                                          AS n_inputs,
    sum(sc)                                                           AS sc_pass,
    round(100.0 * sum(sc) / count(*), 2)                              AS sc_pct,
    sum(coalesce(ic, 0))                                              AS ic_pass,
    round(100.0 * sum(coalesce(ic, 0)) / count(*), 2)                 AS ic_pct,
    sum(CASE WHEN coalesce(ic, 0) = 1 OR empty_cnt >= 2 THEN 1 ELSE 0 END) AS ac_pass,
    round(100.0 * sum(CASE WHEN coalesce(ic, 0) = 1 OR empty_cnt >= 2 THEN 1 ELSE 0 END) / count(*), 2) AS ac_pct
FROM sc_flags
LEFT JOIN ic_flags     USING (custom_id)
LEFT JOIN empty_flags  USING (custom_id);

Running query in 'duckdb:////workspace/ICOCO.duckdb'

n_inputs,sc_pass,sc_pct,ic_pass,ic_pct,ac_pass,ac_pct
17017,16746,98.41,16535,97.17,16898,99.3


In [9]:
%%sql
-- TASK B · Taxonomic Extraction: SC / IC / AC over N = (SELECT count(*) FROM silver.v_scientific_runs)/3 inputs
WITH canon AS (
    SELECT custom_id, run_idx,
        CASE WHEN parsed_kind = 'list'
             THEN to_json(list_sort(value_list))
             WHEN len(coalesce(norm_str, '')) = 0 THEN '[]'
             ELSE to_json(list_sort(string_to_array(coalesce(norm_str, ''), '')))
        END AS canon
    FROM silver.v_scientific_runs
),
per_row AS (
    SELECT custom_id, max(c) AS max_run_freq
    FROM (SELECT custom_id, canon, count(*) AS c FROM canon GROUP BY custom_id, canon)
    GROUP BY custom_id
),
sc_flags AS (
    SELECT custom_id, CASE WHEN max_run_freq >= 2 THEN 1 ELSE 0 END AS sc
    FROM per_row
),
elems AS (
    SELECT custom_id, elem
    FROM (
        SELECT custom_id, parsed_kind, value_list, norm_str
        FROM silver.v_scientific_runs
    ) p
    CROSS JOIN UNNEST(
        CASE WHEN p.parsed_kind = 'list' THEN p.value_list
             WHEN len(coalesce(p.norm_str, '')) = 0 THEN []
             ELSE string_to_array(coalesce(p.norm_str, ''), '') END
    ) AS t(elem)
),
elem_counts AS (
    SELECT custom_id, elem, count(*) AS cnt FROM elems GROUP BY custom_id, elem
),
ic_flags AS (
    SELECT custom_id, max(CASE WHEN cnt >= 2 AND elem <> '' THEN 1 ELSE 0 END) AS ic
    FROM elem_counts GROUP BY custom_id
),
empty_flags AS (
    SELECT custom_id,
           sum(CASE WHEN (parsed_kind = 'list' AND len(value_list) = 0)
                     OR (parsed_kind = 'string' AND len(coalesce(norm_str, '')) = 0)
                    THEN 1 ELSE 0 END) AS empty_cnt
    FROM silver.v_scientific_runs GROUP BY custom_id
)
SELECT
    count(*)                                                          AS n_inputs,
    sum(sc)                                                           AS sc_pass,
    round(100.0 * sum(sc) / count(*), 2)                              AS sc_pct,
    sum(coalesce(ic, 0))                                              AS ic_pass,
    round(100.0 * sum(coalesce(ic, 0)) / count(*), 2)                 AS ic_pct,
    sum(CASE WHEN coalesce(ic, 0) = 1 OR empty_cnt >= 2 THEN 1 ELSE 0 END) AS ac_pass,
    round(100.0 * sum(CASE WHEN coalesce(ic, 0) = 1 OR empty_cnt >= 2 THEN 1 ELSE 0 END) / count(*), 2) AS ac_pct
FROM sc_flags
LEFT JOIN ic_flags     USING (custom_id)
LEFT JOIN empty_flags  USING (custom_id);

Running query in 'duckdb:////workspace/ICOCO.duckdb'

n_inputs,sc_pass,sc_pct,ic_pass,ic_pct,ac_pass,ac_pct
4280,3524,82.34,2626,61.36,4226,98.74


## 4. Gold aggregation (majority vote) — `silver.v_*_gold`

The final output of each task is the *majority-vote* list: keep **only** the candidates that
appear in **≥ 2 of the 3 runs** (legacy `gold_data_create`). These views are what grounding
is evaluated on; `len(final_list) > 0` ⇔ the input "has output" (Task A: 16,524, Task B: 2,415).

In [10]:
%%sql
CREATE OR REPLACE VIEW silver.v_ingredient_gold AS
WITH elems AS (
    SELECT custom_id, unnest(value_list) AS elem
    FROM silver.v_ingredient_runs WHERE parsed_kind = 'list'
),
counts AS (
    SELECT custom_id, elem, count(*) AS cnt FROM elems GROUP BY custom_id, elem
),
gold AS (
    SELECT custom_id, list(elem) AS final_list
    FROM counts WHERE cnt >= 2
    GROUP BY custom_id
)
SELECT * FROM gold;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [11]:
%%sql
CREATE OR REPLACE VIEW silver.v_scientific_gold AS
WITH elems AS (
    SELECT custom_id, unnest(value_list) AS elem
    FROM silver.v_scientific_runs WHERE parsed_kind = 'list'
),
counts AS (
    SELECT custom_id, elem, count(*) AS cnt FROM elems GROUP BY custom_id, elem
),
gold AS (
    SELECT custom_id, list(elem) AS final_list
    FROM counts WHERE cnt >= 2
    GROUP BY custom_id
)
SELECT * FROM gold;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [12]:
%%sql
SELECT 'Task A' AS task, count(*) AS n_inputs,
       sum(CASE WHEN len(final_list) > 0 THEN 1 ELSE 0 END) AS with_output
FROM silver.v_ingredient_gold
UNION ALL
SELECT 'Task B', count(*),
       sum(CASE WHEN len(final_list) > 0 THEN 1 ELSE 0 END)
FROM silver.v_scientific_gold;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

task,n_inputs,with_output
Task A,16524,16524
Task B,2415,2415


## 5. Queryability — Task A (USDA grounding)

**Reference database.** The legacy pipeline matched against `food_master_data.csv`, which is
exactly `FoodUSDA_latest.foundation_food` (469 rows) **∪** `FoodUSDA_legacy.food` (7,793 rows)
= **8,262** food records. We rebuild the same universe from the silver layer
(`silver.v_ref_food_full`) and also expose the mission-strict variant `silver.v_ref_food_foundation`
(foundation_food only).

**Queryability rule.** A gold candidate $o$ is queryable iff $\text{norm}(o)$ is a *substring*
of a lower-cased reference description (legacy `norm(o) in norm(desc)`), skipping empty strings.

In [13]:
%%sql
CREATE OR REPLACE VIEW silver.v_ref_food_foundation AS
SELECT
    fdc_id::VARCHAR          AS fdc_id,
    data_type::VARCHAR       AS data_type,
    description::VARCHAR     AS description,
    lower(description::VARCHAR) AS lower_desc
FROM silver.FoodUSDA_latest_food
WHERE data_type = 'foundation_food';

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [14]:
%%sql
-- the full reference universe used by the paper (foundation_food(latest) + legacy = 8,262)
CREATE OR REPLACE VIEW silver.v_ref_food_full AS
SELECT fdc_id, data_type, description, lower_desc
FROM silver.v_ref_food_foundation
UNION ALL
SELECT fdc_id::VARCHAR, data_type::VARCHAR, description::VARCHAR, lower(description::VARCHAR)
FROM silver.FoodUSDA_legacy_food;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [15]:
%%sql
SELECT 'v_ref_food_full' AS view_name, count(*) AS rows FROM silver.v_ref_food_full
UNION ALL SELECT 'v_ref_food_foundation', count(*) FROM silver.v_ref_food_foundation;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

view_name,rows
v_ref_food_full,8262
v_ref_food_foundation,469


In [16]:
%%sql
-- Task A · USDA full paper universe (foundation latest + legacy)
WITH valid AS (
    SELECT custom_id, final_list
    FROM silver.v_ingredient_gold
    WHERE len(final_list) > 0
),
flagged AS (
    SELECT v.custom_id,
           max(CASE WHEN e.elem <> ''
                     AND EXISTS (SELECT 1 FROM silver.v_ref_food_full r
                                 WHERE r.lower_desc LIKE '%' || e.elem || '%')
                    THEN 1 ELSE 0 END) AS is_queryable
    FROM valid v
    CROSS JOIN UNNEST(v.final_list) AS e(elem)
    GROUP BY v.custom_id
)
SELECT
    count(*)             AS with_output,
    sum(is_queryable)    AS queryable,
    round(100.0 * sum(is_queryable) / count(*), 2) AS queryable_pct
FROM flagged;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

with_output,queryable,queryable_pct
16524,11765,71.2


In [17]:
%%sql
-- Task A · USDA foundation_food ONLY (mission-strict variant)
WITH valid AS (
    SELECT custom_id, final_list
    FROM silver.v_ingredient_gold
    WHERE len(final_list) > 0
),
flagged AS (
    SELECT v.custom_id,
           max(CASE WHEN e.elem <> ''
                     AND EXISTS (SELECT 1 FROM silver.v_ref_food_foundation r
                                 WHERE r.lower_desc LIKE '%' || e.elem || '%')
                    THEN 1 ELSE 0 END) AS is_queryable
    FROM valid v
    CROSS JOIN UNNEST(v.final_list) AS e(elem)
    GROUP BY v.custom_id
)
SELECT
    count(*)             AS with_output,
    sum(is_queryable)    AS queryable,
    round(100.0 * sum(is_queryable) / count(*), 2) AS queryable_pct
FROM flagged;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

with_output,queryable,queryable_pct
16524,7604,46.02


## 6. Queryability — Task B (GBIF Species Matching)

The legacy notebook resolved every unique scientific-name candidate through the
**GBIF Species Matching API** (`https://api.gbif.org/v1/species/match?name=…`) and marked a
name as *queryable* iff `matchType != 'NONE'` (EXACT / FUZZY / HIGHERRANK all count).

To keep this notebook **reproducible** the results are cached in
`data/GBIF_query/gbif_match_cache.json` (+ an SQL-friendly `gbif_species_match.jsonl`). The
cell below only hits the network for names **not already cached**; when the cache is complete
it runs fully offline. It then persists the cache into the DuckDB table `silver.gbif_species_match`.

In [18]:
# 6.1 Build the cache from the DB; fetch only what is missing (offline when complete)
import json, os, requests
from concurrent.futures import ThreadPoolExecutor, as_completed

CACHE_FILE = '/workspace/data/GBIF_query/gbif_match_cache.json'
JSONL_FILE = '/workspace/data/GBIF_query/gbif_species_match.jsonl'

# 1) read candidate names straight from the gold view (SQL already ran above)
result = %sql SELECT DISTINCT elem::VARCHAR AS scientific_name \
              FROM (SELECT unnest(final_list) AS elem FROM silver.v_scientific_gold) \
              ORDER BY scientific_name
names = [row[0] for row in result]
print('unique candidates to ground:', len(names))

# 2) load the existing cache
cache = {}
if os.path.exists(CACHE_FILE):
    cache = json.load(open(CACHE_FILE))
todo = [n for n in names if n not in cache]
print('already cached:', len(names) - len(todo), '| to fetch:', len(todo))

HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) ScientificMatchApp/1.0'}

def fetch(name):
    url = f"https://api.gbif.org/v1/species/match?name={requests.utils.quote(str(name))}"
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        if r.status_code == 200:
            return name, (r.json().get('matchType') != 'NONE')
    except Exception:
        pass
    return name, None

fails = 0
if todo:
    with ThreadPoolExecutor(max_workers=8) as ex:
        futs = [ex.submit(fetch, n) for n in todo]
        for done, f in enumerate(as_completed(futs), 1):
            name, ok = f.result()
            if ok is None:
                fails += 1
            cache[name] = bool(ok)
            if done % 300 == 0:
                print(f'  fetched {done}/{len(todo)} (fails={fails})', flush=True)
    json.dump(cache, open(CACHE_FILE, 'w'), indent=2, ensure_ascii=False)
    print('saved cache ->', CACHE_FILE, '| fails:', fails)
else:
    print('cache complete — no API calls made')

# 3) regenerate the SQL-friendly JSONL (idempotent)
with open(JSONL_FILE, 'w', encoding='utf-8') as f:
    for name in names:
        f.write(json.dumps({'scientific_name': name,
                            'is_queryable': bool(cache.get(name, False))}, ensure_ascii=False) + '\n')
print('jsonl ready ->', JSONL_FILE)

Running query in 'duckdb:////workspace/ICOCO.duckdb'

unique candidates to ground: 1581
already cached: 1581 | to fetch: 0
cache complete — no API calls made
jsonl ready -> /workspace/data/GBIF_query/gbif_species_match.jsonl


In [19]:
%%sql
-- materialise the GBIF cache into DuckDB (idempotent)
CREATE OR REPLACE TABLE silver.gbif_species_match AS
SELECT
    scientific_name::VARCHAR AS scientific_name,
    is_queryable::BOOLEAN    AS is_queryable
FROM read_json_auto('/workspace/data/GBIF_query/gbif_species_match.jsonl');

SELECT count(*) AS cached_names,
       sum(CASE WHEN is_queryable THEN 1 ELSE 0 END) AS matched_names
FROM silver.gbif_species_match;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

cached_names,matched_names
1581,1294


In [20]:
%%sql
-- Task B grounding (GBIF) — 2,253 / 2,415 expected (93.29%)
WITH valid AS (
    SELECT custom_id, final_list
    FROM silver.v_scientific_gold
    WHERE len(final_list) > 0
),
flagged AS (
    SELECT v.custom_id,
           max(CASE WHEN e.elem <> ''
                     AND EXISTS (SELECT 1 FROM silver.gbif_species_match g
                                 WHERE g.scientific_name = e.elem AND g.is_queryable)
                    THEN 1 ELSE 0 END) AS is_queryable
    FROM valid v
    CROSS JOIN UNNEST(v.final_list) AS e(elem)
    GROUP BY v.custom_id
)
SELECT
    count(*)             AS with_output,
    sum(is_queryable)    AS queryable,
    round(100.0 * sum(is_queryable) / count(*), 2) AS queryable_pct
FROM flagged;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

with_output,queryable,queryable_pct
2415,2253,93.29


## 7. Final verdict

| # | Task | Metric | Paper claim | Computed cell | Verdict |
|---|---|---|---|---|---|
| A1 | Ingredient | SC  | 98.41% | §3.1 | ✅ |
| A2 | Ingredient | IC  | 97.17% | §3.1 | ✅ |
| A3 | Ingredient | AC  | 99.30% | §3.1 | ✅ |
| A4 | Ingredient | Grounding (USDA) | 71.20% (11,765/16,524) | §5 | ✅ |
| B1 | Taxonomic | SC  | 82.34% | §3.2 | ✅ |
| B2 | Taxonomic | IC  | 61.36% | §3.2 | ✅ |
| B3 | Taxonomic | AC  | 98.74% | §3.2 | ✅ |
| B4 | Taxonomic | Grounding (GBIF) | 93.29% (2,253/2,415) | §6 | ✅ |

## 8. Discrepancies worth knowing about (details in `confirm_note.md`)

1. **`16,254` is a typo** — the true denominator is **16,524** (Task A inputs whose
   majority-vote output is non-empty): 11,765 / 16,524 = **71.20%**.
2. **"Restrict USDA to `foundation_food` only" does *not* reproduce 71.20%.** The paper's
   11,765 queryable rows are matched against **foundation_food(latest) ∪ legacy (8,262
   records)**. On foundation_food **only** (469 records) the same inputs yield **7,604 (46.02%)**.
3. **SC is "≥ 2/3 identical outputs"** (paper text; confirmed by the legacy code) — *not* "all 3
   non-empty identical": empty lists agree with each other and the printed 98.41% / 82.34% are
   reproduced only under the ≥ 2/3 rule.
4. **AC uses "≥ 2/3 of the runs empty"** as the empty-validity condition (98.74% is reproduced
   only if 2 empty runs + 1 non-empty run also recover coverage), not strictly "all 3 empty".
5. Truncated LLM cells (trailing `\n`, dangling quotes…) are handled by the legacy parser as
   *raw strings*; the quirky `sorted()`/`json.dumps` and string→character expansions inside SC /
   IC are part of the legacy code and are exactly what makes the printed numbers reproducible.

All cells are idempotent (`CREATE OR REPLACE` / `IF NOT EXISTS`); the notebook is fully re-runnable. ✅